# Sentiment Analysis API - Colab Integration

This notebook demonstrates how to run the sentiment analysis API in Google Colab and test it with various text inputs.

In [ ]:
# Install required packages in Colab
!pip install fastapi uvicorn pydantic scikit-learn pandas numpy joblib nltk textblob

In [ ]:
# Mount Google Drive (if using Colab and models are stored there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import joblib
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from typing import List, Dict
import threading
import time
import requests

In [ ]:
# Download NLTK data if needed
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

In [ ]:
# Create API application
app = FastAPI(
    title="Sentiment Analysis API",
    description="API for analyzing sentiment of text reviews",
    version="1.0.0"
)

# Pydantic models for request/response
class TextInput(BaseModel):
    text: str

class BatchTextInput(BaseModel):
    texts: List[str]

class SentimentResponse(BaseModel):
    text: str
    sentiment: str
    confidence: float
    probabilities: Dict[str, float]

class BatchSentimentResponse(BaseModel):
    results: List[SentimentResponse]

In [ ]:
# Text preprocessing function
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocess_text(text):
    """Preprocess text for model input."""
    if not text:
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return ' '.join(tokens)

print("Text preprocessing function defined!")

In [ ]:
# Load model and vectorizer (replace with actual file paths)
# For demo purposes, we'll create a sample model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Create sample training data
sample_data = [
    ("This product is amazing! I love it so much.", "positive"),
    ("Not what I expected. Quality could be better.", "negative"),
    ("Decent product for the price. Works as expected.", "neutral"),
    ("Terrible experience. Would not recommend.", "negative"),
    ("Outstanding quality and fast shipping!", "positive"),
    ("Great value for money, highly recommended!", "positive"),
    ("Poor build quality, broke after one week.", "negative"),
    ("Average product, nothing special but okay.", "neutral")
]

# Prepare data
texts = [preprocess_text(text) for text, _ in sample_data]
labels = [label for _, label in sample_data]

# Train vectorizer and model
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts)

model = MultinomialNB()
model.fit(X, labels)

# Global variables for the loaded model
loaded_model = model
loaded_vectorizer = vectorizer

print("Sample model trained and loaded successfully!")
print(f"Classes: {model.classes_}")

In [ ]:
# Define API endpoints
@app.get("/")
async def root():
    """Root endpoint with API information."""
    return {
        "message": "Sentiment Analysis API",
        "version": "1.0.0",
        "endpoints": {
            "/predict": "POST - Analyze sentiment of single text",
            "/predict_batch": "POST - Analyze sentiment of multiple texts",
            "/health": "GET - Health check"
        }
    }

@app.get("/health")
async def health_check():
    """Health check endpoint."""
    return {"status": "healthy", "model_loaded": loaded_model is not None}

@app.post("/predict", response_model=SentimentResponse)
async def predict_sentiment(input_data: TextInput):
    """Predict sentiment for a single text."""
    try:
        # Preprocess the text
        processed_text = preprocess_text(input_data.text)
        
        # Vectorize the text
        text_vector = loaded_vectorizer.transform([processed_text])
        
        # Make prediction
        prediction = loaded_model.predict(text_vector)[0]
        probabilities = loaded_model.predict_proba(text_vector)[0]
        
        # Get confidence score
        confidence = float(max(probabilities))
        
        # Create probabilities dictionary
        prob_dict = {}
        for i, class_label in enumerate(loaded_model.classes_):
            prob_dict[class_label] = float(probabilities[i])
        
        return SentimentResponse(
            text=input_data.text,
            sentiment=prediction,
            confidence=confidence,
            probabilities=prob_dict
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error processing text: {str(e)}")

@app.post("/predict_batch", response_model=BatchSentimentResponse)
async def predict_sentiment_batch(input_data: BatchTextInput):
    """Predict sentiment for multiple texts."""
    try:
        results = []
        
        for text in input_data.texts:
            # Preprocess the text
            processed_text = preprocess_text(text)
            
            # Vectorize the text
            text_vector = loaded_vectorizer.transform([processed_text])
            
            # Make prediction
            prediction = loaded_model.predict(text_vector)[0]
            probabilities = loaded_model.predict_proba(text_vector)[0]
            
            # Get confidence score
            confidence = float(max(probabilities))
            
            # Create probabilities dictionary
            prob_dict = {}
            for i, class_label in enumerate(loaded_model.classes_):
                prob_dict[class_label] = float(probabilities[i])
            
            results.append(SentimentResponse(
                text=text,
                sentiment=prediction,
                confidence=confidence,
                probabilities=prob_dict
            ))
        
        return BatchSentimentResponse(results=results)
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error processing texts: {str(e)}")

print("API endpoints defined successfully!")

In [ ]:
# Function to run the API server in Colab
import nest_asyncio
from pyngrok import ngrok

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Install pyngrok if not available
!pip install pyngrok

# Set up ngrok tunnel (optional - for external access)
# ngrok.set_auth_token("your_ngrok_token")  # Replace with your token
# public_url = ngrok.connect(8000)
# print(f"Public URL: {public_url}")

print("Ready to start API server!")

In [ ]:
# Start the API server (run this cell to start the server)
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

## Testing the API

After starting the server, you can test it using the following cells or by visiting the API documentation at `http://localhost:8000/docs`

In [ ]:
# Test the API (run in a separate notebook cell after starting the server)
import requests
import json

# Test single prediction
def test_single_prediction(text):
    try:
        response = requests.post(
            "http://localhost:8000/predict",
            json={"text": text},
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            print(f"Text: {result['text']}")
            print(f"Sentiment: {result['sentiment']}")
            print(f"Confidence: {result['confidence']:.3f}")
            print(f"Probabilities: {result['probabilities']}")
        else:
            print(f"Error: {response.status_code} - {response.text}")
    except Exception as e:
        print(f"Error: {e}")

# Example usage
print("Testing single predictions:")
print("=" * 50)
test_single_prediction("This product is absolutely amazing!")
print()
test_single_prediction("I hate this product, it's terrible.")
print()
test_single_prediction("It's an okay product, nothing special.")

In [ ]:
# Test batch prediction
def test_batch_prediction(texts):
    try:
        response = requests.post(
            "http://localhost:8000/predict_batch",
            json={"texts": texts},
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            for i, prediction in enumerate(result['results'], 1):
                print(f"Result {i}:")
                print(f"  Text: {prediction['text']}")
                print(f"  Sentiment: {prediction['sentiment']} (confidence: {prediction['confidence']:.3f})")
                print()
        else:
            print(f"Error: {response.status_code} - {response.text}")
    except Exception as e:
        print(f"Error: {e}")

# Example batch testing
batch_texts = [
    "Great product, highly recommended!",
    "Poor quality, not worth the money.",
    "Average product, meets expectations."
]

print("Testing batch prediction:")
print("=" * 50)
test_batch_prediction(batch_texts)

## API Usage Instructions

### Running in Google Colab

1. **Install dependencies**: Run the first few cells to install required packages
2. **Load models**: The notebook creates a sample model for demonstration
3. **Start the API**: Run the server cell to start the FastAPI application
4. **Test the API**: Use the provided test cells or create your own

### API Endpoints

- **GET /**: Root endpoint with API information
- **GET /health**: Health check endpoint
- **POST /predict**: Analyze sentiment of a single text
- **POST /predict_batch**: Analyze sentiment of multiple texts

### Response Format

```json
{
  "text": "I love this product!",
  "sentiment": "positive",
  "confidence": 0.95,
  "probabilities": {
    "positive": 0.95,
    "negative": 0.03,
    "neutral": 0.02
  }
}
```

### Notes

- Replace the sample model with your trained model by loading from files
- Use ngrok for external access to the API
- The API includes automatic request/response validation using Pydantic
- Interactive API documentation is available at `/docs` endpoint